In [ ]:
!pip install -q transformers datasets sacrebleu accelerate sentencepiece evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.0 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate


In [ ]:
from google.colab import files

uploaded = files.upload()
json_path = list(uploaded.keys())[0]
print(f"Uploaded file: {json_path}")


Saving UN.json to UN.json
Uploaded file: UN.json


In [ ]:
with open(json_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
df = df.rename(columns={"english": "en", "arabic": "ar"})
df = df[["ar", "en"]]

df["ar"] = df["ar"].astype(str).str.strip()
df["en"] = df["en"].astype(str).str.strip()
df = df.dropna(subset=["ar", "en"])
df = df[(df["ar"].str.len() > 3) & (df["en"].str.len() > 3)]
df = df.drop_duplicates(subset=["ar", "en"]).reset_index(drop=True)

print(f"Total pairs after cleaning: {len(df)}")
df.head()


Total pairs after cleaning: 7834


,ar,en
0,١ - اعتمدت الجمعية العامة في دورتها السابعة وا...,"1. At its forty-fourth session, in 1992, the G..."
1,""" ٤ - تحيط علما مع التقدير بالفصل الثاني من تق...",""" 4. Takes note with appreciation of chapter I..."
2,١ - تقدم استراليا التعليقات التالية على تقرير ...,1. Australia provides the following comments o...
3,٣ - قدمت استراليا في البيان الذي أدلت به في أث...,3. In its intervention during the debate on th...
4,٥ - توافق استراليا على الاستنتاج الذي أعرب عنه...,5. Australia agrees with the conclusion of the...


In [ ]:
SRC_LANG = "ar"
TGT_LANG = "en"

if SRC_LANG == "ar" and TGT_LANG == "en":
    MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-ar-en"
elif SRC_LANG == "en" and TGT_LANG == "ar":
    MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-en-ar"
else:
    raise ValueError("Set SRC_LANG/TGT_LANG to ar->en or en->ar")

OUTPUT_DIR = f"./opus-mt-{SRC_LANG}-{TGT_LANG}-un-finetuned"

tokenizer = MarianTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = MarianMTModel.from_pretrained(MODEL_CHECKPOINT)

print(f"Loaded pretrained checkpoint: {MODEL_CHECKPOINT}")


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Loaded pretrained checkpoint: Helsinki-NLP/opus-mt-ar-en


In [ ]:
HARD_CAP = 512

def token_len(text):
    return len(tokenizer(text, truncation=False)["input_ids"])

df["ar_len"] = df["ar"].apply(token_len)
df["en_len"] = df["en"].apply(token_len)

print("قبل حذف الـ outliers:")
print(df[["ar_len", "en_len"]].describe())

df["max_len"] = df[["ar_len", "en_len"]].max(axis=1)

Q1 = df["max_len"].quantile(0.25)
Q3 = df["max_len"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = max(1, Q1 - 1.5 * IQR)
upper_bound = Q3 + 1.5 * IQR

before_count = len(df)
df = df[(df["max_len"] >= lower_bound) & (df["max_len"] <= upper_bound)].reset_index(drop=True)
after_count = len(df)

print(f"تم حذف {before_count - after_count} صف ({(before_count - after_count) / before_count:.1%}) بوصفهم outliers (IQR)")
print(f"نطاق الطول المقبول (IQR): {lower_bound:.0f} - {upper_bound:.0f} توكن")

p90_len = int(np.percentile(df["max_len"], 90))
MAX_LENGTH = min(HARD_CAP, int(np.ceil(p90_len / 8.0) * 8))

print(f"\nP90 لطول الفقرات = {p90_len} توكن")
print(f"MAX_LENGTH النهائي = {MAX_LENGTH} توكن")

before_hardcut = len(df)
would_be_truncated = df[df["max_len"] > MAX_LENGTH]
df = df[df["max_len"] <= MAX_LENGTH].reset_index(drop=True)
after_hardcut = len(df)

print(f"\nتم حذف {before_hardcut - after_hardcut} صف إضافي كانوا هيتقطعوا عند MAX_LENGTH={MAX_LENGTH}")
print(f"الداتا سيت النهائي بعد كل عمليات التنظيف: {after_hardcut} صف")

df = df.drop(columns=["ar_len", "en_len", "max_len"])


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (654 > 512). Running this sequence through the model will result in indexing errors


قبل حذف الـ outliers:
            ar_len       en_len
count  7834.000000  7834.000000
mean    204.049400   358.612331
std     228.669135   410.155248
min       5.000000    11.000000
25%      43.000000    71.000000
50%     113.000000   191.000000
75%     281.000000   496.000000
max    1480.000000  2722.000000
تم حذف 532 صف (6.8%) بوصفهم outliers (IQR)
نطاق الطول المقبول (IQR): 1 - 1134 توكن

P90 لطول الفقرات = 716 توكن
MAX_LENGTH النهائي = 512 توكن

تم حذف 1360 صف إضافي كانوا هيتقطعوا عند MAX_LENGTH=512
الداتا سيت النهائي بعد كل عمليات التنظيف: 5942 صف


In [ ]:
n = len(df)
train_end = int(n * 0.90)
val_end = int(n * 0.95)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

raw_datasets


DatasetDict({
    train: Dataset({
        features: ['ar', 'en'],
        num_rows: 5347
    })
    validation: Dataset({
        features: ['ar', 'en'],
        num_rows: 297
    })
    test: Dataset({
        features: ['ar', 'en'],
        num_rows: 298
    })
})

In [ ]:
def preprocess(batch):
    inputs = batch[SRC_LANG]
    targets = batch[TGT_LANG]

    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


Map:   0%|          | 0/5347 [00:00<?, ? examples/s]

Map:   0%|          | 0/297 [00:00<?, ? examples/s]

Map:   0%|          | 0/298 [00:00<?, ? examples/s]

In [ ]:
sacrebleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu
1,0.844298,0.823560,56.242942
2,0.706164,0.812736,57.211654
3,0.672141,0.814616,57.250280
4,0.645514,0.813507,57.157536
5,0.651118,0.813634,57.058242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=1675, training_loss=0.7070216084950006, metrics={'train_runtime': 954.6231, 'train_samples_per_second': 28.006, 'train_steps_per_second': 1.755, 'total_flos': 1857415090470912.0, 'train_loss': 0.7070216084950006, 'epoch': 5.0})

In [ ]:
test_results = trainer.predict(tokenized_datasets["test"])
print("Test set BLEU:", test_results.metrics["test_bleu"])
print("Full test metrics:", test_results.metrics)


Test set BLEU: 74.12270791798593
Full test metrics: {'test_loss': 0.5439870357513428, 'test_bleu': 74.12270791798593, 'test_runtime': 103.1012, 'test_samples_per_second': 2.89, 'test_steps_per_second': 0.184}


In [ ]:
preds = test_results.predictions
if isinstance(preds, tuple):
    preds = preds[0]

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
source_texts = test_df[SRC_LANG].tolist()
reference_texts = test_df[TGT_LANG].tolist()

n_show = min(10, len(decoded_preds))
comparison_df = pd.DataFrame({
    f"source_{SRC_LANG}": source_texts[:n_show],
    "model_translation": decoded_preds[:n_show],
    f"reference_{TGT_LANG}": reference_texts[:n_show],
})
comparison_df


,source_ar,model_translation,reference_en
0,- استخدام الأسلحة والتخلص من المتفجرات؛,- Use of weapons and disposal of explosives;,- weapons firing and explosive ordnance disposal;
1,- جلسات التزويد بالمعلومات المتعلقة بالاتفاقات...,- Briefings on agreements signed by the parties.,- briefings on any agreements signed between p...
2,٣٣ - يشترك المواطنون الهنغاريون في الوقت الحال...,33. Hungarian citizens are currently involved ...,"33. At the present time, Hungarian citizens ar..."
3,٣٦ - تنسق وزارة الدفاع جميع أنواع التدريب للأف...,36. The Ministry of Defence coordinates all ty...,36. The Department of Defence coordinates all ...
4,ويتم التدريب على مستوى السريات والكتائب. ٣٩ - ...,"39. During training exercises, emphasis is pla...",Training takes place at the company and battal...
5,- ما يفعل وما يترك في الثقافة اللبنانية؛,- What it does and what is left behind in Leba...,"- the "" do ' s and don ' ts "" of Lebanese cult..."
6,- اجراءات الحوادث والاجلاء الطبي في قوة الأمم ...,- Accident and medical evacuation procedures a...,- UNIFIL accident and medical evacuation proce...
7,٤٠ - توفر القوات المسلحة الايطالية التدريب لأف...,40. The Italian Armed Forces provide training ...,40. The Italian Armed Forces provides training...
8,- جلسات للتزويد بالمعلومات المتعلقة بالحالة ال...,- Briefings on the political and social situat...,- briefings on the political and social situat...
9,٤٤ - تقدم جمهورية كوريا برنامجا تدريبيا لفرقته...,44. The Republic of Korea is providing a train...,44. The Republic of Korea provides a training ...


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./opus-mt-ar-en-un-finetuned


In [ ]:
def translate(text, model, tokenizer, max_length=MAX_LENGTH):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
    generated = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

sample_text = "تؤكد الجمعية العامة من جديد التزامها بحقوق الإنسان."
print(translate(sample_text, model, tokenizer))


The General Assembly reaffirms its commitment to human rights.
